# Phase 6 — Memory Retention Model Training (v2: Log-Transform)

## Why Synthetic Data?

A newly deployed spaced-repetition system has no longitudinal review history spanning weeks or months. We bootstrap by generating 5 000 synthetic review sessions whose half-life targets are derived from well-established cognitive-science principles (Ebbinghaus forgetting curve, desirable-difficulty effect, hesitation-as-weakness proxy). As real user reviews accumulate, the model can be retrained incrementally.

## Feature Engineering

| Feature | Cognitive rationale |
|---|---|
| `review_count` | More exposures = stronger trace |
| `correct_count` | Successful retrievals actively strengthen memories |
| `incorrect_count` | Failures reset or weaken the memory trace |
| `avg_response_time_s` | Slow responses signal weaker encoding |
| `days_since_last_review` | Spacing effect: longer gaps before recall strengthen retention |
| `question_difficulty` | Inverse SM-2 easiness — harder cards decay faster |

## Log-Transform Motivation

The raw half-life target spans 0.1–365 days (highly right-skewed, std ≈ 101 d). Training a GBR directly on this scale penalises large-h errors with MSE heavily, distorting predictions for the majority of cards (67 % have h ≤ 7 days). Transforming with `log1p` compresses the tail so the model learns relative error uniformly across the full range. At inference, `expm1` inverts the prediction back to days.

## Model Iteration Log

| Version | Target | MAE (days) | RMSE (days) |
|---|---|---|---|
| v1 | raw half_life | 7.1749 | 16.6582 |
| v2 | log1p(half_life) | 6.8071 | 19.5937 |

In [1]:
import os, joblib
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

In [2]:
np.random.seed(42)
N = 5000
review_count           = np.random.randint(1, 21, size=N)
correct_count          = np.array([np.random.randint(0, r+1) for r in review_count])
incorrect_count        = review_count - correct_count
avg_response_time_s    = np.random.uniform(1.0, 10.0, size=N)
days_since_last_review = np.random.uniform(0.1, 30.0, size=N)
question_difficulty    = np.random.uniform(0.0, 1.0, size=N)
h_base             = 2.5 * (1.0 - question_difficulty) + 0.5
accuracy           = correct_count / review_count
expansion_factor   = (1.5 ** correct_count) * (0.6 ** incorrect_count)
hesitation_penalty = 0.9 ** np.maximum(0.0, avg_response_time_s - 2.0)
spacing_multiplier = 1.0 + 0.05 * days_since_last_review * accuracy
h = h_base * expansion_factor * hesitation_penalty * spacing_multiplier * (1.0 + 0.1 * review_count)
h = np.clip(h, 0.1, 365.0)
noise     = np.random.lognormal(0.0, 0.15, size=N)
half_life = np.clip(h * noise, 0.1, 365.0)
X = np.column_stack((review_count, correct_count, incorrect_count,
                     avg_response_time_s, days_since_last_review, question_difficulty))
print(f'Generated {N:,} synthetic review sessions.')

Generated 5,000 synthetic review sessions.


In [3]:
X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X, half_life, test_size=0.2, random_state=42)
# Apply log1p transform to training target only
y_train_log = np.log1p(y_train_raw)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

Train: 4000 | Test: 1000


In [4]:
gbr = GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
gbr.fit(X_train, y_train_log)
print('GBR trained on log1p(half_life).')

GBR trained on log1p(half_life).


In [5]:
y_pred_log  = gbr.predict(X_test)
y_pred_days = np.maximum(0.1, np.expm1(y_pred_log))  # invert + floor
mae  = mean_absolute_error(y_test_raw, y_pred_days)
rmse = root_mean_squared_error(y_test_raw, y_pred_days)
print('Evaluation on held-out test set (real day-scale after expm1 inversion):')
print(f'  Model v1 (raw target):    MAE = 7.1749 d | RMSE = 16.6582 d')
print(f'  Model v2 (log-transform): MAE = {mae:.4f} d | RMSE = {rmse:.4f} d')

Evaluation on held-out test set (real day-scale after expm1 inversion):
  Model v1 (raw target): MAE = 7.1749 d | RMSE = 16.6582 d
  Model v2 (log-transform): MAE = 6.8071 d | RMSE = 19.5937 d


In [6]:
MODEL_PATH = os.path.normpath(os.path.join(os.getcwd(), '..', 'ml', 'retention_model.joblib'))
joblib.dump(gbr, MODEL_PATH)
print(f'Exported: {MODEL_PATH}  ({os.path.getsize(MODEL_PATH):,} bytes)')

Exported: D:\final-year-project\backend\ml\retention_model.joblib  (256,376 bytes)
